# Course direction around the flash, by altitude (supplemental figure)

Occupancy of course direction (heading) vs. time for flies that stayed low (z < 0.25 m)
or high (z > 0.25 m) for the whole trajectory, in both split-flow configurations.

Ported from the course-direction heatmaps of the earlier `f4_trajectory_characteristics.ipynb`
(same selection, bins, colour scale and styling). Built on `supp_course_direction_template.svg`
(layout boxes + hand-made artwork: schematics, example trajectories, labels); the committed,
published version is `output_figs/supp_course_direction.svg`.

Note: the original notebook read `..._top_laminar_09092024.hdf`; the data on Dryad (and here) is
`..._top_laminar_09082024.hdf`. All counts match the published figure except configuration 2,
low, flash (197 here vs 211 published).

In [ ]:
import shutil
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import figurefirst as fifi

In [ ]:
# Matplotlib text (tick labels, n=) in DejaVu Serif, as in the published figure;
# the Helvetica labels are text in the template.
plt.rcParams['font.serif'] = ['Times'] + plt.rcParams['font.serif']
plt.rcParams['text.usetex'] = False
plt.rcParams["ps.usedistiller"] = 'xpdf'
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.weight'] = 'normal'
plt.rcParams["mathtext.fontset"] = 'cm'

In [ ]:
TEMPLATE_SVG = 'supp_course_direction_template.svg'
FIGURE_SVG = 'supp_course_direction.svg'
shutil.copy(TEMPLATE_SVG, FIGURE_SVG)
layout = fifi.svg_to_axes.FigureLayout(FIGURE_SVG, autogenlayers=True, make_mplfigures=True, hide_layers=[])

## Load data

In [ ]:
DATA_DIR = '../../../Data/Experimental_Fly_Data/'
bottom_on_df = pd.read_hdf(DATA_DIR + 'all_nights_new_processing_protocol_bottom_laminar_09092024.hdf')
top_on_df = pd.read_hdf(DATA_DIR + 'all_nights_new_processing_protocol_top_laminar_09082024.hdf')

## Selection

In [ ]:
def split_flash_sham_by_altitude(df):
    '''Flash / sham trials, split into flies that stayed below or above 0.25 m.'''
    flash = df[df.duration > 0]
    sham = df[df.duration == 0]
    low = lambda d: d.z.max() < 0.25
    up = lambda d: d.z.min() > 0.25
    sel = lambda x, f: x.groupby('obj_id_unique').filter(f)
    return {'flash_low': sel(flash, low), 'flash_up': sel(flash, up),
            'sham_low': sel(sham, low), 'sham_up': sel(sham, up)}


def drop_duplicates(df):
    return df.groupby('obj_id_unique').filter(lambda d: np.any(d.duplicate_obj_id_bool) == False)


def sham_start_in_center(df, z_max):
    '''Sham trials that start in the central volume (as for the flash trigger).'''
    start = df[df['time stamp'] == 0]
    ids = start.groupby('obj_id_unique').filter(
        lambda d: (d['x'].between(-.35, 0.35)) & (d['y'].between(-0.1, 0.1)) & (d['z'].between(0.05, z_max))).obj_id_unique
    return df[df.obj_id_unique.isin(list(ids))]


bottom_on = split_flash_sham_by_altitude(bottom_on_df)
top_on = split_flash_sham_by_altitude(top_on_df)

## Plot

In [ ]:
T_EDGES = np.arange(-.500, 3.010, .010)
HEADING_EDGES = np.linspace(-np.pi, np.pi, 181)
VMAX = 0.04
YTICKLABELS = [r'-$\pi$', r'-$\frac{\pi}{2}$', r'$0$', r'$\frac{\pi}{2}$', r'$\pi$']


def occupancy(df):
    '''Heading histogram per time bin, normalised within each time bin.'''
    counts, _, _ = np.histogram2d(df['time stamp'] / 1000, df['heading'], bins=(T_EDGES, HEADING_EDGES), density=True)
    return counts.T / np.sum(counts, axis=1)


def plot_flash_sham(fig_name, ax_names, flash_df, sham_df, first_column):
    X, Y = np.meshgrid(T_EDGES, HEADING_EDGES)
    for ax_name, df, span_color, span_alpha in ((ax_names[0], flash_df, 'red', 0.1),
                                                (ax_names[1], sham_df, 'grey', 0.2)):
        ax = layout.axes[(fig_name, ax_name)]
        ax.pcolormesh(X, Y, occupancy(df), vmin=0, vmax=VMAX, cmap='bone_r', shading='flat', rasterized=True)
        fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'], yticks=[-np.pi, -np.pi/2, 0, np.pi/2, np.pi],
                                         xticks=[-.5, 0, .68, 3], spine_locations={'left': 5, 'bottom': 5},
                                         tick_length=3, linewidth=.5)
        ax.set_yticklabels(YTICKLABELS if first_column else [''] * 5)
        bottom_row_labels = first_column and ax_name == ax_names[1]
        ax.set_xticklabels(['-0.5', '0', '0.68', '3'] if bottom_row_labels else [''] * 4)
        ax.axvspan(0, .675, color=span_color, edgecolor=None, alpha=span_alpha)
        ax.set_aspect(.3)
        fifi.mpl_functions.set_fontsize(ax, 6)
        ax.text(x=3, y=3.14, horizontalalignment='left', verticalalignment='top',
                s='n=' + str(df.obj_id_unique.nunique()), fontsize=5, fontstyle='italic', rotation=90,
                bbox=dict(facecolor='white', edgecolor='white', pad=0.05, alpha=0.7))
    layout.append_figure_to_layer(layout.figures[fig_name], fig_name, cleartarget=True)


# configuration 1 (bottom half on): low flies (a), high flies (c)
plot_flash_sham('a', ('A', 'B'), drop_duplicates(bottom_on['flash_low']),
                sham_start_in_center(drop_duplicates(bottom_on['sham_low']), 0.35), first_column=True)
plot_flash_sham('c', ('E', 'F'), drop_duplicates(bottom_on['flash_up']),
                sham_start_in_center(drop_duplicates(bottom_on['sham_up']), 0.35), first_column=False)
# configuration 2 (top half on): high flies (b), low flies (d)
plot_flash_sham('b', ('C', 'D'), drop_duplicates(top_on['flash_up']),
                sham_start_in_center(drop_duplicates(top_on['sham_up']), 0.4), first_column=False)
plot_flash_sham('d', ('G', 'H'), drop_duplicates(top_on['flash_low']),
                sham_start_in_center(drop_duplicates(top_on['sham_low']), 0.4), first_column=False)

## Colour bar

In [ ]:
ax = layout.axes[('CB', 'cb')]
sm = mpl.cm.ScalarMappable(cmap='bone_r', norm=mpl.colors.Normalize(vmin=0, vmax=VMAX))
layout.figures['CB'].colorbar(sm, cax=ax, orientation='vertical')
fifi.mpl_functions.adjust_spines(ax, ['right'], yticks=[0, VMAX], spine_locations={'right': 1}, tick_length=1, linewidth=0.5)
ax.set_yticklabels(['', ''])  # 0.0% / 0.04% are text in the template
fifi.mpl_functions.set_fontsize(ax, 6)
layout.append_figure_to_layer(layout.figures['CB'], 'CB', cleartarget=True)

In [ ]:
layout.write_svg(FIGURE_SVG)